In [1]:
!pip install -q segmentation-models-pytorch rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.4 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import rasterio
from tqdm import tqdm
import segmentation_models_pytorch as smp

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

DATA_DIR = "/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data"

Device: cuda


In [3]:
def load_split(path):
    with open(path) as f:
        return [l.strip() for l in f if l.strip()]

train_ids = load_split(f"{DATA_DIR}/split/train.txt")
val_ids   = load_split(f"{DATA_DIR}/split/val.txt")

all_ids = train_ids + val_ids

test_ids = [f.replace("_image.tif","")
            for f in os.listdir(f"{DATA_DIR}/prediction/image")]

print(len(all_ids), len(test_ids))

69 19


In [4]:
def preprocess(img):
    hh, hv = img[0], img[1]
    green, red, nir, swir = img[2], img[3], img[4], img[5]

    eps = 1e-6

    ndwi  = (green - nir) / (green + nir + eps)
    mndwi = (green - swir) / (green + swir + eps)
    ndvi  = (nir - red) / (nir + red + eps)
    sar_diff = hh - hv

    bands = [hh, hv, green, red, nir, swir, ndwi, mndwi, ndvi, sar_diff]

    normed = []
    for b in bands:
        p2, p98 = np.percentile(b, [2,98])
        b = np.clip(b, p2, p98)
        b = (b - p2)/(p98 - p2 + eps)
        normed.append(b)

    return np.stack(normed).astype(np.float32)

In [5]:
class FloodDataset(Dataset):
    def __init__(self, ids, mode="train"):
        self.ids = ids
        self.mode = mode

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        pid = self.ids[idx]

        path = f"{DATA_DIR}/prediction/image/{pid}_image.tif" if self.mode=="test" \
               else f"{DATA_DIR}/image/{pid}_image.tif"

        with rasterio.open(path) as src:
            img = src.read().astype(np.float32)

        img = preprocess(img)

        if self.mode != "test":
            with rasterio.open(f"{DATA_DIR}/label/{pid}_label.tif") as src:
                mask = src.read(1).astype(np.int64)

            return torch.from_numpy(img), torch.from_numpy(mask)

        else:
            return torch.from_numpy(img), pid

In [6]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=10,
    classes=3,
    decoder_attention_type="scse"
).to(device)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

In [7]:
weights = torch.tensor([0.3, 5.0, 1.0]).to(device)

ce = nn.CrossEntropyLoss(weight=weights)
dice = smp.losses.DiceLoss(mode="multiclass")

def loss_fn(logits, targets):
    return 0.5 * ce(logits, targets) + 0.5 * dice(logits, targets)

In [8]:
train_loader = DataLoader(FloodDataset(all_ids,"train"), batch_size=4, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

for epoch in range(20):
    model.train()
    total_loss = 0

    for imgs, masks in tqdm(train_loader):
        imgs, masks = imgs.to(device), masks.to(device)

        optimizer.zero_grad()
        loss = loss_fn(model(imgs), masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch} Loss: {total_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), "/kaggle/working/model2.pth")

100%|██████████| 18/18 [00:20<00:00,  1.16s/it]


Epoch 0 Loss: 0.8878


100%|██████████| 18/18 [00:11<00:00,  1.53it/s]


Epoch 1 Loss: 0.7736


100%|██████████| 18/18 [00:11<00:00,  1.53it/s]


...

Epoch 17 Loss: 0.4120


100%|██████████| 18/18 [00:11<00:00,  1.53it/s]


Epoch 18 Loss: 0.4064


100%|██████████| 18/18 [00:11<00:00,  1.51it/s]


Epoch 19 Loss: 0.3875


In [9]:
# =========================
# TRAIN FRIEND MODEL (FAST REBUILD)
# =========================

import segmentation_models_pytorch as smp

model1 = smp.UnetPlusPlus(
    encoder_name="timm-efficientnet-b5",
    encoder_weights="imagenet",
    in_channels=10,
    classes=3,
    decoder_attention_type="scse"
).to(device)

weights = torch.tensor([0.3, 5.0, 1.0]).to(device)
ce = nn.CrossEntropyLoss(weight=weights)
dice = smp.losses.DiceLoss(mode="multiclass")

def loss_fn(logits, targets):
    return 0.4 * ce(logits, targets) + 0.6 * dice(logits, targets)

train_loader = DataLoader(FloodDataset(all_ids,"train"), batch_size=4, shuffle=True)

optimizer = torch.optim.AdamW(model1.parameters(), lr=2e-4)

# 🔥 ONLY TRAIN 10 EPOCHS (ENOUGH FOR ENSEMBLE)
for epoch in range(10):
    model1.train()
    total_loss = 0

    for imgs, masks in tqdm(train_loader):
        imgs, masks = imgs.to(device), masks.to(device)

        optimizer.zero_grad()
        loss = loss_fn(model1(imgs), masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[Model1] Epoch {epoch} Loss: {total_loss/len(train_loader):.4f}")

torch.save(model1.state_dict(), "/kaggle/working/best.pth")

print("✅ Friend model saved as best.pth")

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

100%|██████████| 18/18 [00:23<00:00,  1.28s/it]


[Model1] Epoch 0 Loss: 0.7617


100%|██████████| 18/18 [00:22<00:00,  1.27s/it]


...

[Model1] Epoch 7 Loss: 0.3999


100%|██████████| 18/18 [00:23<00:00,  1.29s/it]


[Model1] Epoch 8 Loss: 0.4056


100%|██████████| 18/18 [00:23<00:00,  1.29s/it]


[Model1] Epoch 9 Loss: 0.4082
✅ Friend model saved as best.pth


In [10]:
# =========================
# SAVE MODEL2 PROBS
# =========================

model.load_state_dict(torch.load("/kaggle/working/model2.pth"))
model.eval()

for pid in test_ids:
    with rasterio.open(f"{DATA_DIR}/prediction/image/{pid}_image.tif") as src:
        img = preprocess(src.read().astype(np.float32))

    img = torch.from_numpy(img).unsqueeze(0).float().to(device)

    with torch.no_grad():
        probs = torch.softmax(model(img), dim=1)[0].cpu().numpy()

    np.save(f"/kaggle/working/{pid}_model2.npy", probs)

print("✅ Model2 probs saved")

✅ Model2 probs saved


In [11]:
# =========================
# FINAL ENSEMBLE
# =========================

model1.load_state_dict(torch.load("/kaggle/working/best.pth"))
model1.eval()

def rle(mask):
    pixels=(mask==1).astype(np.uint8).flatten(order="F")
    pixels=np.concatenate([[0],pixels,[0]])
    runs=np.where(pixels[1:]!=pixels[:-1])[0]+1
    runs[1::2]-=runs[::2]
    return " ".join(map(str,runs)) if len(runs)>0 else "0 0"

rows = []

for pid in tqdm(test_ids):

    with rasterio.open(f"{DATA_DIR}/prediction/image/{pid}_image.tif") as src:
        img = preprocess(src.read().astype(np.float32))

    img_tensor = torch.from_numpy(img).unsqueeze(0).float().to(device)

    with torch.no_grad():
        probs1 = torch.softmax(model1(img_tensor), dim=1)[0].cpu().numpy()

    probs2 = np.load(f"/kaggle/working/{pid}_model2.npy")

    # 🔥 ENSEMBLE
    final_probs = 0.8 * probs1 + 0.2 * probs2
    flood_prob = final_probs[1]

    # 🔥 TRY THIS THRESHOLD
    mask = (flood_prob > 0.33).astype(np.uint8)

    pred = final_probs.argmax(0)

    rows.append({"id": pid, "rle_mask": rle(pred)})

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/submission_ensemblev7.csv", index=False)

print("🔥 FINAL SUBMISSION READY")

100%|██████████| 19/19 [00:03<00:00,  5.86it/s]

🔥 FINAL SUBMISSION READY
